# AMIA Public Challenge 2026 — DenseNet121 + FPN Faster R-CNN

This notebook is a corrected, Kaggle-ready pipeline for the supplied AMIA Public Challenge 2026 dataset.

### Dataset structure used by this notebook
```text
/kaggle/input/competitions/amia-public-challenge-2026/
├── train/
│   ├── train/                  # training PNG images
│   ├── train.csv
│   ├── test.csv
│   ├── img_size.csv
│   └── sample_submission.csv
└── test/
    └── test/                   # test PNG images
```

### Main corrections
- Uses the actual nested Kaggle CSV/image paths.
- Uses `img_size.csv` to convert original annotation coordinates to the supplied 1024×1024 PNG coordinate system.
- Splits by `image_id` to prevent annotation leakage.
- Keeps train/validation datasets separate.
- Fuses multi-radiologist boxes while calculating agreement from actual radiologist support.
- Uses a **custom DenseNet121 backbone + FeaturePyramidNetwork** with direct dense-block outputs:
  `256, 512, 1024, 1024` channels.
- Uses exactly one RPN anchor group per FPN feature map.
- Creates the detector DataLoader explicitly before training.
- Handles class 14 (`No finding`) separately from abnormality detection classes 0–13.
- Uses the sample submission's image order for the final CSV.


In [ ]:
# ============================================================
# 0. Optional package check
# ============================================================
import sys, subprocess, importlib.util

if importlib.util.find_spec("cv2") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "opencv-python-headless"
    ])

print("Package check complete.")


In [ ]:
# ============================================================
# 1. Imports, reproducibility and device
# ============================================================
import os
import random
import warnings

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Dataset configuration

The supplied Kaggle dataset has this structure:

```text
/kaggle/input/competitions/amia-public-challenge-2026/
├── train/
│   ├── train/
│   │   └── *.png
│   ├── train.csv
│   ├── test.csv
│   ├── img_size.csv
│   └── sample_submission.csv
└── test/
    └── test/
        └── *.png
```


In [ ]:
# ============================================================
# 2. Exact paths for the supplied AMIA dataset
# ============================================================

ROOT = "/kaggle/input/competitions/amia-public-challenge-2026"

TRAIN_IMG_DIR = f"{ROOT}/train/train"
TEST_IMG_DIR = f"{ROOT}/test/test"

TRAIN_CSV = "/kaggle/input/competitions/amia-public-challenge-2026/train.csv"
TEST_CSV = "/kaggle/input/competitions/amia-public-challenge-2026/test.csv"
IMG_SIZE_CSV = "/kaggle/input/competitions/amia-public-challenge-2026/img_size.csv"
SAMPLE_SUBMISSION = "/kaggle/input/competitions/amia-public-challenge-2026/sample_submission.csv"

WORK_DIR = "/kaggle/working"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

# -------------------- training knobs --------------------
# None = all available training images.
# Use 500 first to test Run All end-to-end.
MAX_TRAIN_IMAGES = 500

CLF_EPOCHS = 1
DET_EPOCHS = 1

CLF_BATCH = 16
DET_BATCH = 2

CLF_LR = 1e-4
DET_LR = 2e-4

IMG_SIZE_CLF = 384
IMG_SIZE_DET = 512

# AMIA supplies PNG images resized to 1024×1024.
# train.csv boxes are expressed in the original image coordinate system,
# so they are converted to this canonical coordinate system first.
ANNOTATION_IMAGE_SIZE = 1024

NUM_WORKERS = 2
PIN_MEMORY = DEVICE.type == "cuda"

# ImageNet weights are recommended. If Kaggle cannot download them,
# the model automatically falls back to random initialization.
USE_PRETRAINED = True

DET_SCORE_THRESHOLD = 0.05
DET_NMS_THRESHOLD = 0.4
MAX_DETECTIONS = 100

NUM_FINDINGS = 14
NO_FINDING_ID = 14
NUM_CLASSES_DET = NUM_FINDINGS + 1  # + background

print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TEST_CSV)
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("TEST_IMG_DIR:", TEST_IMG_DIR)

for p in [TRAIN_CSV, TEST_CSV, IMG_SIZE_CSV, TRAIN_IMG_DIR, TEST_IMG_DIR]:
    print(f"{p} ->", os.path.exists(p))

assert os.path.exists(TRAIN_CSV), "train.csv not found"
assert os.path.exists(TEST_CSV), "test.csv not found"
assert os.path.exists(IMG_SIZE_CSV), "img_size.csv not found"
assert os.path.isdir(TRAIN_IMG_DIR), "Training image directory not found"
assert os.path.isdir(TEST_IMG_DIR), "Test image directory not found"


In [ ]:
# ============================================================
# 3. Load CSV files and inspect schema
# ============================================================
train_raw = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
img_size_df = pd.read_csv(IMG_SIZE_CSV)

print("train.csv shape:", train_raw.shape)
print("test.csv shape:", test_df.shape)
print("img_size.csv shape:", img_size_df.shape)

print("\nTrain columns:")
print(train_raw.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

print("\nImage-size columns:")
print(img_size_df.columns.tolist())

display(train_raw.head())


In [ ]:
# ============================================================
# 4. Validate the AMIA schema
# ============================================================
required_train_cols = {
    "image_id", "class_name", "class_id", "rad_id",
    "x_min", "y_min", "x_max", "y_max"
}

missing = required_train_cols - set(train_raw.columns)
assert not missing, f"Missing train.csv columns: {missing}"

assert {"image_id", "dim0", "dim1"}.issubset(img_size_df.columns)

print("Schema OK.")
print("Unique train images:", train_raw["image_id"].nunique())
print("Unique radiologists:", train_raw["rad_id"].nunique())
print("Class IDs:", sorted(train_raw["class_id"].dropna().astype(int).unique().tolist()))


## 3. Image loading and preprocessing

The supplied AMIA data uses PNG images. The loader also supports DICOM as a fallback.

For detector training, bounding boxes are transformed together with the letterboxed image. Classifier tensors are ImageNet-normalized; detector tensors remain in [0,1] because torchvision Faster R-CNN applies ImageNet normalization internally.


In [ ]:
# ============================================================
# 5. Image utilities
# ============================================================
try:
    import pydicom
    HAVE_PYDICOM = True
except Exception:
    HAVE_PYDICOM = False

def find_image_path(img_dir, image_id):
    image_id = str(image_id)

    candidates = [
        os.path.join(img_dir, image_id),
        os.path.join(img_dir, image_id + ".png"),
        os.path.join(img_dir, image_id + ".jpg"),
        os.path.join(img_dir, image_id + ".jpeg"),
        os.path.join(img_dir, image_id + ".dcm"),
        os.path.join(img_dir, image_id + ".dicom"),
    ]

    for p in candidates:
        if os.path.isfile(p):
            return p

    raise FileNotFoundError(
        f"Image not found for {image_id} in {img_dir}"
    )


def load_image_any(path):
    ext = os.path.splitext(path)[1].lower()

    if ext in (".dcm", ".dicom"):
        if not HAVE_PYDICOM:
            raise ImportError("pydicom is required for DICOM input.")

        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)

        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = arr.max() - arr

        arr -= arr.min()

        if arr.max() > 0:
            arr /= arr.max()

        arr = (arr * 255).astype(np.uint8)

        return cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)

    img = cv2.imread(path, cv2.IMREAD_COLOR)

    if img is None:
        raise FileNotFoundError(path)

    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def letterbox_resize(img, size):
    h, w = img.shape[:2]

    scale = min(size / h, size / w)

    nh = max(1, int(round(h * scale)))
    nw = max(1, int(round(w * scale)))

    resized = cv2.resize(
        img,
        (nw, nh),
        interpolation=cv2.INTER_LINEAR
    )

    canvas = np.zeros(
        (size, size, 3),
        dtype=np.uint8
    )

    top = (size - nh) // 2
    left = (size - nw) // 2

    canvas[
        top:top + nh,
        left:left + nw
    ] = resized

    return canvas, scale, left, top


IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)


def to_tensor(img_uint8):
    x = img_uint8.astype(np.float32) / 255.0

    x = (
        x - IMAGENET_MEAN
    ) / IMAGENET_STD

    x = np.transpose(x, (2, 0, 1))

    return torch.from_numpy(x).float()

def to_tensor_raw(img_uint8):
    """Convert uint8 RGB image to [0,1] tensor without normalization.

    Faster R-CNN applies ImageNet normalization internally through
    GeneralizedRCNNTransform, so detector inputs must NOT be normalized
    here.
    """
    x = img_uint8.astype(np.float32) / 255.0
    x = np.transpose(x, (2, 0, 1))
    return torch.from_numpy(x).float()


print("Image utilities ready.")


In [ ]:
# ============================================================
# 6. Quick image sanity check
# ============================================================

from PIL import Image
import os

# Make sure train_raw exists
if "train_raw" not in globals():
    train_raw = pd.read_csv(TRAIN_CSV)

sample_id = str(train_raw["image_id"].iloc[0])

# find_image_path expects: (img_dir, image_id)
sample_path = find_image_path(
    TRAIN_IMG_DIR,
    sample_id
)

print("Sample image ID   :", sample_id)
print("Sample image path :", sample_path)

assert os.path.exists(sample_path), \
    f"Image does not exist: {sample_path}"

img = Image.open(sample_path).convert("RGB")

print("Image size:", img.size)
print("Image mode:", img.mode)

display(img)

In [ ]:
# ============================================================
# 6B. Verify supplied image size and coordinate scaling
# ============================================================
# The AMIA challenge provides PNGs resized to 1024×1024, while
# img_size.csv records the original dimensions used by train.csv.

check_ids = train_raw["image_id"].drop_duplicates().head(10).tolist()
loaded_shapes = []
for image_id in check_ids:
    p = find_image_path(TRAIN_IMG_DIR, image_id)
    loaded_shapes.append(load_image_any(p).shape[:2])

print("Sample loaded image shapes:", loaded_shapes)
print("Target annotation coordinate size:", ANNOTATION_IMAGE_SIZE)

if all(tuple(s) == (ANNOTATION_IMAGE_SIZE, ANNOTATION_IMAGE_SIZE) for s in loaded_shapes):
    print("✅ Supplied PNGs are 1024×1024 for the checked samples.")
else:
    print("⚠️ Some checked images are not 1024×1024; the notebook will map boxes using actual image dimensions.")

max_original_coord = float(
    train_raw[["x_min", "y_min", "x_max", "y_max"]].max().max()
)
print("Maximum raw annotation coordinate:", max_original_coord)
print("img_size.csv is used to normalize these original coordinates before WBF.")


## 4. Multi-radiologist annotation fusion

Overlapping boxes from different radiologists are fused before detector training. `"No finding"` is not treated as an object box.


In [ ]:
# ============================================================
# 7. Weighted Boxes Fusion utilities
# ============================================================
def box_iou_one_to_many(box, boxes):
    boxes = np.asarray(boxes, dtype=np.float64)

    if len(boxes) == 0:
        return np.zeros(0, dtype=np.float64)

    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])

    inter = (
        np.clip(x2 - x1, 0, None) *
        np.clip(y2 - y1, 0, None)
    )

    area_a = (
        max(0.0, box[2] - box[0]) *
        max(0.0, box[3] - box[1])
    )

    area_b = (
        np.clip(boxes[:, 2] - boxes[:, 0], 0, None) *
        np.clip(boxes[:, 3] - boxes[:, 1], 0, None)
    )

    union = area_a + area_b - inter

    return inter / np.clip(union, 1e-9, None)


def simple_wbf(
    boxes_list,
    scores_list,
    labels_list,
    iou_thr=0.5
):
    all_boxes = []
    all_scores = []
    all_labels = []

    for b, s, l in zip(
        boxes_list,
        scores_list,
        labels_list
    ):
        if len(b):
            all_boxes.append(
                np.asarray(b, dtype=np.float64)
            )
            all_scores.append(
                np.asarray(s, dtype=np.float64)
            )
            all_labels.append(
                np.asarray(l, dtype=np.int64)
            )

    if not all_boxes:
        return (
            np.zeros((0, 4)),
            np.zeros(0),
            np.zeros(0, dtype=np.int64)
        )

    boxes = np.concatenate(all_boxes)
    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels)

    fused_boxes = []
    fused_scores = []
    fused_labels = []

    for cls in np.unique(labels):
        idx = np.where(labels == cls)[0]

        cb = boxes[idx]
        cs = scores[idx]

        order = np.argsort(-cs)

        cb = cb[order]
        cs = cs[order]

        used = np.zeros(
            len(cb),
            dtype=bool
        )

        for i in range(len(cb)):
            if used[i]:
                continue

            ious = box_iou_one_to_many(
                cb[i],
                cb
            )

            cluster = np.where(
                (ious >= iou_thr) &
                (~used)
            )[0]

            used[cluster] = True

            weights = cs[cluster]

            fused_box = (
                cb[cluster] *
                weights[:, None]
            ).sum(axis=0) / max(
                weights.sum(),
                1e-9
            )

            fused_score = float(
                weights.mean()
            )

            fused_boxes.append(fused_box)
            fused_scores.append(fused_score)
            fused_labels.append(int(cls))

    return (
        np.asarray(fused_boxes).reshape(-1, 4),
        np.asarray(fused_scores),
        np.asarray(fused_labels, dtype=np.int64)
    )


def estimate_annotation_agreement(
    fused_box,
    fused_label,
    group,
    image_h,
    image_w,
    iou_thr=0.5
):
    """Fraction of distinct radiologists supporting the fused finding."""
    if "rad_id" not in group.columns:
        return 1.0

    rad_ids = (
        group["rad_id"]
        .dropna()
        .unique()
        .tolist()
    )

    if not rad_ids:
        return 1.0

    support = 0

    for rad in rad_ids:
        rg = group[
            (group["rad_id"] == rad) &
            (group["class_id"] == fused_label)
        ]

        if rg.empty:
            continue

        boxes = rg[
            ["x_min", "y_min", "x_max", "y_max"]
        ].values.astype(float)

        boxes[:, [0, 2]] /= image_w
        boxes[:, [1, 3]] /= image_h

        if np.max(
            box_iou_one_to_many(
                fused_box,
                boxes
            )
        ) >= iou_thr:
            support += 1

    return float(
        support / len(rad_ids)
    )


In [ ]:
# ============================================================
# 8. Merge/fuse training annotations
# ============================================================
# IMPORTANT: train.csv coordinates use the ORIGINAL image dimensions
# from img_size.csv. The supplied PNGs are 1024×1024, so we first
# normalize using the original dimensions and finally store fused
# boxes in the 1024×1024 PNG coordinate system.

img_lookup = (
    img_size_df
    .drop_duplicates("image_id")
    .set_index("image_id")
)

def merge_multireader_boxes(
    train_df,
    img_lookup,
    iou_thr=0.5,
    target_size=1024
):
    out_rows = []

    for image_id, group in tqdm(
        train_df.groupby("image_id"),
        desc="Fusing annotations"
    ):
        if image_id in img_lookup.index:
            image_h = float(img_lookup.loc[image_id, "dim0"])
            image_w = float(img_lookup.loc[image_id, "dim1"])
        else:
            image_h = float(target_size)
            image_w = float(target_size)

        # Work in normalized original-image coordinates for WBF.
        findings = group[group["class_id"] != NO_FINDING_ID].copy()

        if findings.empty:
            out_rows.append({
                "image_id": image_id,
                "class_id": NO_FINDING_ID,
                "agreement": 1.0,
                "x_min": 0.0,
                "y_min": 0.0,
                "x_max": 1.0,
                "y_max": 1.0
            })
            continue

        rad_ids = (
            findings["rad_id"].dropna().unique().tolist()
            if "rad_id" in findings.columns
            else [0]
        )

        boxes_list = []
        scores_list = []
        labels_list = []

        for rad in rad_ids:
            rg = (
                findings[findings["rad_id"] == rad]
                if "rad_id" in findings.columns
                else findings
            )

            b = rg[["x_min", "y_min", "x_max", "y_max"]].values.astype(float)

            # Normalize using the ORIGINAL dimensions.
            b[:, [0, 2]] /= max(image_w, 1.0)
            b[:, [1, 3]] /= max(image_h, 1.0)

            b = np.clip(b, 0.0, 1.0)

            boxes_list.append(b.tolist())
            scores_list.append([1.0] * len(b))
            labels_list.append(rg["class_id"].astype(int).tolist())

        fb, fs, fl = simple_wbf(
            boxes_list, scores_list, labels_list, iou_thr=iou_thr
        )

        if len(fb) == 0:
            continue

        for box, score, label in zip(fb, fs, fl):
            agreement = estimate_annotation_agreement(
                box, int(label), group, image_h, image_w, iou_thr=iou_thr
            )

            # Convert fused normalized coordinates to the supplied
            # 1024×1024 PNG coordinate system.
            x1 = float(np.clip(box[0] * target_size, 0, target_size - 1))
            y1 = float(np.clip(box[1] * target_size, 0, target_size - 1))
            x2 = float(np.clip(box[2] * target_size, 0, target_size - 1))
            y2 = float(np.clip(box[3] * target_size, 0, target_size - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            out_rows.append({
                "image_id": image_id,
                "class_id": int(label),
                "agreement": float(agreement),
                "x_min": x1,
                "y_min": y1,
                "x_max": x2,
                "y_max": y2
            })

    return pd.DataFrame(out_rows)

# Use a new cache filename so an older, incorrectly scaled cache is never reused.
MERGED_PATH = os.path.join(
    WORK_DIR,
    "train_merged_1024.csv"
)

if os.path.exists(MERGED_PATH):
    train_merged = pd.read_csv(MERGED_PATH)

    required_merged_cols = {
        "image_id", "class_id", "agreement",
        "x_min", "y_min", "x_max", "y_max"
    }

    cache_ok = (
        required_merged_cols.issubset(train_merged.columns)
        and len(train_merged) > 0
        and train_merged[["x_min", "y_min", "x_max", "y_max"]]
            .apply(np.isfinite).all().all()
        and train_merged[["x_min", "y_min", "x_max", "y_max"]]
            .max().max() <= ANNOTATION_IMAGE_SIZE
    )

    if not cache_ok:
        print("Existing merged cache is invalid; rebuilding it.")
        train_merged = merge_multireader_boxes(
            train_raw,
            img_lookup,
            iou_thr=0.5,
            target_size=ANNOTATION_IMAGE_SIZE
        )
        train_merged.to_csv(MERGED_PATH, index=False)
    else:
        print("Reusing:", MERGED_PATH)
else:
    train_merged = merge_multireader_boxes(
        train_raw,
        img_lookup,
        iou_thr=0.5,
        target_size=ANNOTATION_IMAGE_SIZE
    )

    train_merged.to_csv(MERGED_PATH, index=False)
    print("Saved:", MERGED_PATH)

print("Merged rows:", len(train_merged))
print("Merged images:", train_merged["image_id"].nunique())
print("Coordinate range:")
print(train_merged[["x_min", "y_min", "x_max", "y_max"]].describe())
display(train_merged.head())


In [ ]:
# ============================================================
# 9. Class distribution
# ============================================================
class_names = (
    train_raw[
        ["class_id", "class_name"]
    ]
    .drop_duplicates()
    .sort_values("class_id")
)

display(class_names)

plt.figure(figsize=(12, 5))

counts = (
    train_raw["class_id"]
    .value_counts()
    .sort_index()
)

plt.bar(
    counts.index.astype(str),
    counts.values
)

plt.xlabel("class_id")
plt.ylabel("annotation count")
plt.title("Raw annotation distribution")
plt.show()


## 5. Image-level train/validation split

The split is performed on unique `image_id`, not individual annotation rows. This prevents annotations from the same image appearing in both training and validation.


In [ ]:
# ============================================================
# 10. Image-level train/validation split
# ============================================================
all_train_ids = (
    train_merged["image_id"]
    .drop_duplicates()
    .tolist()
)

rng = np.random.default_rng(SEED)
rng.shuffle(all_train_ids)

if MAX_TRAIN_IMAGES is not None:
    all_train_ids = all_train_ids[
        :int(MAX_TRAIN_IMAGES)
    ]

n_val = max(
    1,
    int(round(0.10 * len(all_train_ids)))
)

val_ids = set(
    all_train_ids[:n_val]
)

train_ids = set(
    all_train_ids[n_val:]
)

train_merged_split = train_merged[
    train_merged["image_id"].isin(
        all_train_ids
    )
].copy()

train_df_split = train_merged_split[
    train_merged_split["image_id"].isin(
        train_ids
    )
].copy()

val_df_split = train_merged_split[
    train_merged_split["image_id"].isin(
        val_ids
    )
].copy()

print(
    "Total selected images:",
    len(all_train_ids)
)

print(
    "Train images:",
    len(train_ids)
)

print(
    "Validation images:",
    len(val_ids)
)

print(
    "Train annotation rows:",
    len(train_df_split)
)

print(
    "Validation annotation rows:",
    len(val_df_split)
)


In [ ]:
# ============================================================
# 11. Dataset classes
# ============================================================
class CXRClassifierDataset(Dataset):
    def __init__(
        self,
        df,
        img_dir,
        img_size,
        train=True
    ):
        self.img_dir = img_dir
        self.img_size = img_size
        self.train = train

        items = []

        for image_id, g in df.groupby(
            "image_id"
        ):
            class_ids = (
                g["class_id"]
                .astype(int)
                .tolist()
            )

            normal = all(
                c == NO_FINDING_ID
                for c in class_ids
            )

            multi = np.zeros(
                NUM_FINDINGS,
                dtype=np.float32
            )

            if not normal:
                for c in sorted(
                    set(class_ids)
                ):
                    if (
                        c != NO_FINDING_ID and
                        0 <= c < NUM_FINDINGS
                    ):
                        multi[c] = 1.0

            items.append(
                (
                    image_id,
                    float(not normal),
                    multi
                )
            )

        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        (
            image_id,
            abnormal,
            multi
        ) = self.items[idx]

        path = find_image_path(
            self.img_dir,
            image_id
        )

        img = load_image_any(path)

        canvas, _, _, _ = letterbox_resize(
            img,
            self.img_size
        )

        if self.train:
            if random.random() < 0.30:
                alpha = 1.0 + random.uniform(
                    -0.15,
                    0.15
                )

                beta = random.uniform(
                    -15,
                    15
                )

                canvas = np.clip(
                    canvas.astype(
                        np.float32
                    ) * alpha + beta,
                    0,
                    255
                ).astype(np.uint8)

        return (
            to_tensor(canvas),
            torch.tensor(
                abnormal,
                dtype=torch.float32
            ),
            torch.tensor(
                multi,
                dtype=torch.float32
            ),
            image_id
        )


class CXRDetectionDataset(Dataset):
    def __init__(
        self,
        df,
        img_dir,
        img_size,
        train=True
    ):
        df = df[
            df["class_id"] != NO_FINDING_ID
        ].copy()

        self.img_dir = img_dir
        self.img_size = img_size
        self.train = train

        self.groups = list(
            df.groupby("image_id")
        )

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        image_id, g = self.groups[idx]

        path = find_image_path(
            self.img_dir,
            image_id
        )

        img = load_image_any(path)

        canvas, scale, left, top = letterbox_resize(
            img,
            self.img_size
        )

        boxes = g[
            [
                "x_min",
                "y_min",
                "x_max",
                "y_max"
            ]
        ].values.astype(np.float32).copy()

        # train_merged_1024.csv is stored in the canonical 1024×1024
        # PNG coordinate system. Map it to the actual loaded image size
        # before applying the detector letterbox transform.
        h0, w0 = img.shape[:2]
        boxes[:, [0, 2]] *= float(w0) / ANNOTATION_IMAGE_SIZE
        boxes[:, [1, 3]] *= float(h0) / ANNOTATION_IMAGE_SIZE

        boxes[:, [0, 2]] = (
            boxes[:, [0, 2]] *
            scale + left
        )

        boxes[:, [1, 3]] = (
            boxes[:, [1, 3]] *
            scale + top
        )

        boxes[:, [0, 2]] = np.clip(
            boxes[:, [0, 2]],
            0,
            self.img_size - 1
        )

        boxes[:, [1, 3]] = np.clip(
            boxes[:, [1, 3]],
            0,
            self.img_size - 1
        )

        valid = (
            (boxes[:, 2] > boxes[:, 0] + 1) &
            (boxes[:, 3] > boxes[:, 1] + 1)
        )

        boxes = boxes[valid]

        labels = (
            g["class_id"]
            .values
            .astype(np.int64)[valid]
            + 1
        )

        if "agreement" in g.columns:
            weights = (
                g["agreement"]
                .values
                .astype(np.float32)[valid]
            )
        else:
            weights = np.ones(
                len(labels),
                dtype=np.float32
            )

        target = {
            "boxes": torch.as_tensor(
                boxes,
                dtype=torch.float32
            ),
            "labels": torch.as_tensor(
                labels,
                dtype=torch.int64
            ),
            "weights": torch.as_tensor(
                weights,
                dtype=torch.float32
            ),
            "image_id": image_id
        }

        return (
            to_tensor_raw(canvas),
            target,
            image_id
        )


def detection_collate_fn(batch):
    images, targets, image_ids = zip(*batch)
    return (
        list(images),
        list(targets),
        list(image_ids)
    )

print("Datasets ready.")


In [ ]:
# ============================================================
# 11B. Detector DataLoader
# ============================================================
# The detector dataset class is defined above, but the training
# DataLoader must also be created before the detector smoke test
# and detector training cells are run.

det_train_dataset = CXRDetectionDataset(
    train_df_split,
    TRAIN_IMG_DIR,
    IMG_SIZE_DET,
    train=True
)

det_val_dataset = CXRDetectionDataset(
    val_df_split,
    TRAIN_IMG_DIR,
    IMG_SIZE_DET,
    train=False
)

det_train_loader = DataLoader(
    det_train_dataset,
    batch_size=DET_BATCH,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=detection_collate_fn,
    drop_last=False
)

det_val_loader = DataLoader(
    det_val_dataset,
    batch_size=DET_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=detection_collate_fn,
    drop_last=False
)

print("Detector DataLoader ready.")
print("Detector train images:", len(det_train_dataset))
print("Detector validation images:", len(det_val_dataset))
print("Detector train batches:", len(det_train_loader))
print("Detector validation batches:", len(det_val_loader))

# Quick structural check before Cell 15.5.
if len(det_train_dataset) == 0:
    raise RuntimeError(
        "Detector training dataset is empty. "
        "Check train_df_split and the class filtering."
    )

if len(det_train_loader) == 0:
    raise RuntimeError(
        "Detector training DataLoader is empty."
    )

print("✓ det_train_loader is defined and ready for Cell 15.5")


## 6. DenseNet121 triage classifier

In [ ]:
# ============================================================
# 12. DenseNet121 triage classifier
# ============================================================
def load_densenet121(pretrained=True):
    if pretrained:
        try:
            weights = (
                torchvision.models
                .DenseNet121_Weights
                .DEFAULT
            )

            print(
                "Loading ImageNet-pretrained DenseNet121..."
            )

            return torchvision.models.densenet121(
                weights=weights
            )

        except Exception as e:
            print(
                "[warning] pretrained weights unavailable:",
                e
            )

    print(
        "Using randomly initialized DenseNet121."
    )

    return torchvision.models.densenet121(
        weights=None
    )


class DenseNet121Triage(nn.Module):
    def __init__(
        self,
        pretrained=True,
        dropout_p=0.30
    ):
        super().__init__()

        backbone = load_densenet121(
            pretrained=pretrained
        )

        self.features = backbone.features

        feat_dim = (
            backbone.classifier.in_features
        )

        self.dropout = nn.Dropout(
            dropout_p
        )

        self.binary_head = nn.Linear(
            feat_dim,
            1
        )

        self.multilabel_head = nn.Linear(
            feat_dim,
            NUM_FINDINGS
        )

    def forward(self, x):
        feats = F.relu(
            self.features(x),
            inplace=True
        )

        feats = F.adaptive_avg_pool2d(
            feats,
            (1, 1)
        ).flatten(1)

        feats = self.dropout(feats)

        return {
            "logit": self.binary_head(
                feats
            ).squeeze(1),

            "multi_logits": self.multilabel_head(
                feats
            )
        }


def mc_dropout_predict(
    model,
    x,
    n_mc=10
):
    model.eval()

    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

    probs = []

    with torch.no_grad():
        for _ in range(n_mc):
            out = model(x)

            probs.append(
                torch.sigmoid(
                    out["logit"]
                )
            )

    probs = torch.stack(probs)

    return (
        probs.mean(0),
        probs.std(0)
    )

print("Classifier definition ready.")


In [ ]:
# ============================================================
# 13. Train classifier
# ============================================================
clf_train_ds = CXRClassifierDataset(
    train_df_split,
    TRAIN_IMG_DIR,
    IMG_SIZE_CLF,
    train=True
)

clf_val_ds = CXRClassifierDataset(
    val_df_split,
    TRAIN_IMG_DIR,
    IMG_SIZE_CLF,
    train=False
)

clf_train_loader = DataLoader(
    clf_train_ds,
    batch_size=CLF_BATCH,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

clf_val_loader = DataLoader(
    clf_val_ds,
    batch_size=CLF_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

classifier = DenseNet121Triage(
    pretrained=USE_PRETRAINED,
    dropout_p=0.30
).to(DEVICE)

optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=CLF_LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(CLF_EPOCHS, 1)
)

bce_binary = nn.BCEWithLogitsLoss()
bce_multi = nn.BCEWithLogitsLoss()

amp_enabled = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled
)

best_val_loss = float("inf")

clf_ckpt_path = os.path.join(
    CKPT_DIR,
    "classifier_best.pt"
)

for epoch in range(CLF_EPOCHS):
    classifier.train()

    running = 0.0

    pbar = tqdm(
        clf_train_loader,
        desc=f"Classifier {epoch+1}/{CLF_EPOCHS}"
    )

    for imgs, y_bin, y_multi, _ in pbar:
        imgs = imgs.to(
            DEVICE,
            non_blocking=True
        )

        y_bin = y_bin.to(
            DEVICE,
            non_blocking=True
        )

        y_multi = y_multi.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            enabled=amp_enabled
        ):
            out = classifier(imgs)

            loss_bin = bce_binary(
                out["logit"],
                y_bin
            )

            loss_multi = bce_multi(
                out["multi_logits"],
                y_multi
            )

            loss = (
                loss_bin +
                0.5 * loss_multi
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += loss.item()

        pbar.set_postfix(
            loss=running /
            max(pbar.n, 1)
        )

    scheduler.step()

    classifier.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, y_bin, y_multi, _ in clf_val_loader:
            imgs = imgs.to(
                DEVICE,
                non_blocking=True
            )

            y_bin = y_bin.to(
                DEVICE,
                non_blocking=True
            )

            y_multi = y_multi.to(
                DEVICE,
                non_blocking=True
            )

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=amp_enabled
            ):
                out = classifier(imgs)

                loss = (
                    bce_binary(
                        out["logit"],
                        y_bin
                    )
                    +
                    0.5 * bce_multi(
                        out["multi_logits"],
                        y_multi
                    )
                )

            val_loss += loss.item()

            pred = (
                torch.sigmoid(
                    out["logit"]
                ) >= 0.5
            ).float()

            correct += (
                pred == y_bin
            ).sum().item()

            total += y_bin.numel()

    avg_train = (
        running /
        max(len(clf_train_loader), 1)
    )

    avg_val = (
        val_loss /
        max(len(clf_val_loader), 1)
    )

    val_acc = (
        correct /
        max(total, 1)
    )

    print(
        f"Epoch {epoch+1}: "
        f"train_loss={avg_train:.4f}, "
        f"val_loss={avg_val:.4f}, "
        f"val_accuracy={val_acc:.4f}"
    )

    if avg_val < best_val_loss:
        best_val_loss = avg_val

        torch.save(
            {
                "model":
                    classifier.state_dict(),
                "img_size":
                    IMG_SIZE_CLF,
                "best_val_loss":
                    best_val_loss
            },
            clf_ckpt_path
        )

        print(
            "Saved:",
            clf_ckpt_path
        )

print(
    "Best classifier validation loss:",
    best_val_loss
)


## 7. DenseNet121 + FPN Faster R-CNN

The detector uses a custom DenseNet121 backbone. We explicitly capture the output **after each DenseNet dense block**, before the transition layer changes the channel count:

```text
denseblock1 → 256 channels
denseblock2 → 512 channels
denseblock3 → 1024 channels
denseblock4 → 1024 channels
```

Those four tensors are passed to `FeaturePyramidNetwork(out_channels=256)`.

The FPN therefore has exactly 4 feature maps, and the RPN uses exactly 4 anchor groups:

```text
(16,), (32,), (64,), (128,)
```

This avoids the earlier `BackboneWithFPN` channel mismatch and anchor-count errors.


In [ ]:
# ============================================================
# 14. Detector architecture note
# ============================================================
# The old BackboneWithFPN implementation was intentionally removed.
#
# It captured DenseNet outputs at the wrong points and could cause
# channel mismatches such as:
#   expected 512 channels, got 1024
#
# Cell 15 below is the ONLY detector-construction cell.
# It uses direct dense-block outputs:
#   256, 512, 1024, 1024
# and feeds them into a custom FeaturePyramidNetwork.
print("Old BackboneWithFPN detector skipped; using the robust custom FPN in Cell 15.")


In [ ]:
# ============================================================
# 15. DenseNet121 + FPN + Faster R-CNN
#     ROBUST CUSTOM BACKBONE
# ============================================================

import torch
import torch.nn as nn

from torchvision.models import densenet121
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork


print("=" * 70)
print("Creating DenseNet121 + FPN + Faster R-CNN")
print("=" * 70)


# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)


# ------------------------------------------------------------
# 2. Detector configuration
# ------------------------------------------------------------

if "IMG_SIZE_DET" not in globals():
    IMG_SIZE_DET = 512

if "USE_PRETRAINED" not in globals():
    USE_PRETRAINED = True

print("Detector image size:", IMG_SIZE_DET)
print("Pretrained DenseNet:", USE_PRETRAINED)


# ------------------------------------------------------------
# 3. Load DenseNet121
# ------------------------------------------------------------

if USE_PRETRAINED:

    try:
        from torchvision.models import DenseNet121_Weights

        base_model = densenet121(
            weights=DenseNet121_Weights.DEFAULT
        )

        print("✓ ImageNet pretrained DenseNet121 loaded")

    except Exception as e:

        print("Could not load pretrained weights.")
        print("Reason:", e)

        base_model = densenet121(
            weights=None
        )

else:

    base_model = densenet121(
        weights=None
    )

    print("✓ DenseNet121 created without pretrained weights")


# ------------------------------------------------------------
# 4. Custom DenseNet121 FPN Backbone
# ------------------------------------------------------------

class DenseNet121FPNBackbone(nn.Module):

    def __init__(self, densenet):

        super().__init__()

        self.features = densenet.features

        # DenseNet121 outputs after dense blocks:
        #
        # denseblock1 -> 256 channels
        # denseblock2 -> 512 channels
        # denseblock3 -> 1024 channels
        # denseblock4 -> 1024 channels

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=[
                256,
                512,
                1024,
                1024
            ],
            out_channels=256
        )

        # Faster R-CNN expects this attribute
        self.out_channels = 256


    def forward(self, x):

        features = self.features


        # ----------------------------------------------------
        # Initial DenseNet layers
        # ----------------------------------------------------

        x = features.conv0(x)
        x = features.norm0(x)
        x = features.relu0(x)
        x = features.pool0(x)


        # ----------------------------------------------------
        # Dense Block 1
        # ----------------------------------------------------

        x = features.denseblock1(x)

        c1 = x

        # c1 = 256 channels

        x = features.transition1(x)


        # ----------------------------------------------------
        # Dense Block 2
        # ----------------------------------------------------

        x = features.denseblock2(x)

        c2 = x

        # c2 = 512 channels

        x = features.transition2(x)


        # ----------------------------------------------------
        # Dense Block 3
        # ----------------------------------------------------

        x = features.denseblock3(x)

        c3 = x

        # c3 = 1024 channels

        x = features.transition3(x)


        # ----------------------------------------------------
        # Dense Block 4
        # ----------------------------------------------------

        x = features.denseblock4(x)

        c4 = x

        # c4 = 1024 channels


        # ----------------------------------------------------
        # DenseNet final normalization
        # ----------------------------------------------------

        c4 = features.norm5(c4)

        # IMPORTANT:
        # torch.relu() does not use inplace=False
        c4 = torch.relu(c4)


        # ----------------------------------------------------
        # Send DenseNet feature maps into FPN
        # ----------------------------------------------------

        fpn_input = {

            "stage1": c1,

            "stage2": c2,

            "stage3": c3,

            "stage4": c4

        }


        fpn_output = self.fpn(
            fpn_input
        )

        return fpn_output


# ------------------------------------------------------------
# 5. Create backbone
# ------------------------------------------------------------

backbone = DenseNet121FPNBackbone(
    base_model
)

backbone = backbone.to(
    DEVICE
)


print()
print("✓ Custom DenseNet121 FPN backbone created")


# ------------------------------------------------------------
# 6. Test backbone
# ------------------------------------------------------------

print()
print("=" * 70)
print("Testing DenseNet121 + FPN")
print("=" * 70)


backbone.eval()


with torch.no_grad():

    dummy = torch.zeros(
        1,
        3,
        IMG_SIZE_DET,
        IMG_SIZE_DET,
        device=DEVICE
    )

    features_test = backbone(
        dummy
    )


print()
print("FPN output feature maps:")
print("-" * 70)


for name, feature in features_test.items():

    print(
        f"{name}: {tuple(feature.shape)}"
    )


num_feature_maps = len(
    features_test
)


print()
print(
    "Number of FPN feature maps:",
    num_feature_maps
)


# ------------------------------------------------------------
# 7. Verify FPN channels
# ------------------------------------------------------------

for name, feature in features_test.items():

    assert feature.shape[1] == 256, (
        f"{name} has {feature.shape[1]} channels. "
        f"Expected 256."
    )


print("✓ All FPN feature maps have 256 channels")


# ------------------------------------------------------------
# 8. Anchor configuration
# ------------------------------------------------------------

# We have exactly 4 FPN feature maps.
#
# Therefore we MUST provide exactly 4
# anchor-size groups and 4 aspect-ratio groups.

anchor_sizes = (

    (16,),

    (32,),

    (64,),

    (128,)

)


anchor_aspect_ratios = (

    (0.5, 1.0, 2.0),

    (0.5, 1.0, 2.0),

    (0.5, 1.0, 2.0),

    (0.5, 1.0, 2.0)

)


print()
print("Anchor sizes:")
print(anchor_sizes)

print()
print("Anchor aspect ratios:")
print(anchor_aspect_ratios)


assert len(anchor_sizes) == num_feature_maps

assert len(anchor_aspect_ratios) == num_feature_maps


print()
print("✓ Anchor groups match FPN feature maps")


# ------------------------------------------------------------
# 9. Create Anchor Generator
# ------------------------------------------------------------

anchor_generator = AnchorGenerator(

    sizes=anchor_sizes,

    aspect_ratios=anchor_aspect_ratios

)


print("✓ AnchorGenerator created")


# ------------------------------------------------------------
# 10. Detector classes
# ------------------------------------------------------------

# AMIA has:
#
# 14 abnormality classes
# + background
#
# Faster R-CNN uses:
#
# 0  = background
# 1-14 = detector foreground classes

NUM_CLASSES_DETECTOR = 15


print()
print(
    "Detector classes:",
    NUM_CLASSES_DETECTOR
)


# ------------------------------------------------------------
# 11. Create Faster R-CNN
# ------------------------------------------------------------

detector = FasterRCNN(

    backbone=backbone,

    num_classes=NUM_CLASSES_DETECTOR,

    rpn_anchor_generator=anchor_generator,

    min_size=IMG_SIZE_DET,

    max_size=IMG_SIZE_DET,


    # --------------------------------------------------------
    # RPN settings
    # --------------------------------------------------------

    rpn_pre_nms_top_n_train=2000,

    rpn_pre_nms_top_n_test=1000,

    rpn_post_nms_top_n_train=1000,

    rpn_post_nms_top_n_test=300,

    rpn_nms_thresh=0.7,


    # --------------------------------------------------------
    # Detection settings
    # --------------------------------------------------------

    box_score_thresh=0.05,

    box_nms_thresh=0.5,

    box_detections_per_img=100

)


detector = detector.to(
    DEVICE
)


print()
print("✓ Faster R-CNN created")


# ------------------------------------------------------------
# 12. Final detector verification
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL DETECTOR VERIFICATION")
print("=" * 70)


detector.eval()


with torch.no_grad():

    dummy = torch.zeros(
        1,
        3,
        IMG_SIZE_DET,
        IMG_SIZE_DET,
        device=DEVICE
    )

    final_features = detector.backbone(
        dummy
    )


feature_count = len(
    final_features
)

anchor_count = len(
    detector.rpn.anchor_generator.sizes
)

ratio_count = len(
    detector.rpn.anchor_generator.aspect_ratios
)


print()
print("FPN feature maps    :", feature_count)
print("Anchor groups       :", anchor_count)
print("Aspect-ratio groups :", ratio_count)
print("Detector classes    :", NUM_CLASSES_DETECTOR)


# ------------------------------------------------------------
# 13. Assertions
# ------------------------------------------------------------

assert feature_count == 4, (
    f"Expected 4 FPN feature maps, "
    f"got {feature_count}"
)

assert anchor_count == 4, (
    f"Expected 4 anchor groups, "
    f"got {anchor_count}"
)

assert ratio_count == 4, (
    f"Expected 4 aspect-ratio groups, "
    f"got {ratio_count}"
)

assert feature_count == anchor_count == ratio_count


print()
print("✓ DenseNet121 backbone works")
print("✓ DenseNet121 channels are correct")
print("✓ FPN works")
print("✓ FPN produces 4 feature maps")
print("✓ AnchorGenerator matches FPN")
print("✓ Faster R-CNN created successfully")
print()
print("=" * 70)
print("CELL 15 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# 15.5 Detector Smoke Test
#     Custom FPN + Faster R-CNN
# ============================================================

import torch
from torchvision.ops import MultiScaleRoIAlign


print("=" * 70)
print("15.5 - Faster R-CNN Smoke Test")
print("=" * 70)


# ------------------------------------------------------------
# 1. Check required objects
# ------------------------------------------------------------

assert "detector" in globals(), (
    "detector is not defined. Run Cell 15 first."
)

assert "det_train_loader" in globals(), (
    "det_train_loader is not defined. "
    "Run the detector DataLoader cell first."
)


# ------------------------------------------------------------
# 2. Fix RoI pooling for custom FPN names
# ------------------------------------------------------------

roi_pooler = MultiScaleRoIAlign(
    featmap_names=[
        "stage1",
        "stage2",
        "stage3",
        "stage4"
    ],
    output_size=7,
    sampling_ratio=2
)

detector.roi_heads.box_roi_pool = roi_pooler

print("✓ Custom MultiScaleRoIAlign attached")


# ------------------------------------------------------------
# 3. Get one batch
# ------------------------------------------------------------

detector.train()

smoke_batch = next(
    iter(det_train_loader)
)


# ------------------------------------------------------------
# 4. Inspect batch structure
# ------------------------------------------------------------

print()
print("Batch type:", type(smoke_batch))

if isinstance(smoke_batch, (list, tuple)):

    print(
        "Number of batch elements:",
        len(smoke_batch)
    )

    for i, item in enumerate(smoke_batch):

        print(
            f"  Element {i}: "
            f"type={type(item)}"
        )

else:

    print("Batch is not a list/tuple.")


# ------------------------------------------------------------
# 5. Extract images and targets
# ------------------------------------------------------------

if isinstance(smoke_batch, (list, tuple)):

    if len(smoke_batch) == 2:

        # Standard detection loader:
        # images, targets

        smoke_images = smoke_batch[0]
        smoke_targets = smoke_batch[1]

        smoke_extra = None


    elif len(smoke_batch) >= 3:

        # Your loader returns extra information.
        #
        # Usually:
        # images, targets, image_ids
        #
        # We only need the first two for Faster R-CNN.

        smoke_images = smoke_batch[0]
        smoke_targets = smoke_batch[1]

        smoke_extra = smoke_batch[2:]

        print()
        print(
            "Extra batch information detected:",
            len(smoke_extra)
        )

    else:

        raise ValueError(
            "Batch does not contain enough elements."
        )

else:

    raise TypeError(
        f"Unexpected batch type: {type(smoke_batch)}"
    )


# ------------------------------------------------------------
# 6. Verify images and targets
# ------------------------------------------------------------

print()
print("Images type:", type(smoke_images))
print("Targets type:", type(smoke_targets))


assert isinstance(
    smoke_images,
    (list, tuple)
), (
    "Expected images to be a list/tuple."
)


assert isinstance(
    smoke_targets,
    (list, tuple)
), (
    "Expected targets to be a list/tuple."
)


assert len(smoke_images) == len(smoke_targets), (
    "Number of images and targets does not match."
)


print(
    "Number of images:",
    len(smoke_images)
)


# ------------------------------------------------------------
# 7. Move images to device
# ------------------------------------------------------------

smoke_images = [
    image.to(DEVICE)
    for image in smoke_images
]


# ------------------------------------------------------------
# 8. Move targets to device
# ------------------------------------------------------------

smoke_targets = [

    {
        key: value.to(DEVICE)
        if torch.is_tensor(value)
        else value

        for key, value in target.items()
    }

    for target in smoke_targets

]


# ------------------------------------------------------------
# 9. Inspect targets
# ------------------------------------------------------------

print()
print("Target information:")
print("-" * 70)


for i, target in enumerate(smoke_targets):

    print(
        f"Image {i}: "
        f"{len(target['boxes'])} boxes"
    )

    print(
        "  Labels:",
        target["labels"]
        .detach()
        .cpu()
        .tolist()
    )


# ------------------------------------------------------------
# 10. Validate targets
# ------------------------------------------------------------

for target in smoke_targets:

    assert "boxes" in target
    assert "labels" in target

    boxes = target["boxes"]
    labels = target["labels"]


    assert boxes.ndim == 2
    assert boxes.shape[1] == 4

    assert labels.ndim == 1

    assert len(boxes) == len(labels)


    if len(boxes) > 0:

        assert torch.isfinite(
            boxes
        ).all(), (
            "Bounding boxes contain NaN or Inf."
        )

        assert torch.isfinite(
            labels.float()
        ).all(), (
            "Labels contain NaN or Inf."
        )

        assert (
            boxes[:, 2] > boxes[:, 0]
        ).all(), (
            "Some boxes have x2 <= x1."
        )

        assert (
            boxes[:, 3] > boxes[:, 1]
        ).all(), (
            "Some boxes have y2 <= y1."
        )


print()
print("✓ Target validation passed")


# ------------------------------------------------------------
# 11. Run Faster R-CNN
# ------------------------------------------------------------

print()
print("Running Faster R-CNN forward pass...")
print("This may take a little while on the first run.")


with torch.no_grad():

    smoke_loss_dict = detector(
        smoke_images,
        smoke_targets
    )


# ------------------------------------------------------------
# 12. Display losses
# ------------------------------------------------------------

print()
print("Detector losses:")
print("-" * 70)


total_loss = 0.0


for name, value in smoke_loss_dict.items():

    loss_value = float(
        value.detach().cpu()
    )

    print(
        f"{name:<30}: {loss_value:.6f}"
    )

    assert torch.isfinite(value), (
        f"Non-finite loss detected: {name}"
    )

    total_loss += loss_value


print()
print(
    f"Total loss: {total_loss:.6f}"
)


# ------------------------------------------------------------
# 13. Final verification
# ------------------------------------------------------------

assert len(smoke_loss_dict) > 0

assert torch.isfinite(
    torch.tensor(total_loss)
)


print()
print("=" * 70)
print("✓ SMOKE TEST PASSED")
print("=" * 70)

print()
print("DenseNet121")
print("    ↓")
print("Custom FPN")
print("    ↓")
print("RPN")
print("    ↓")
print("MultiScaleRoIAlign")
print("    ↓")
print("ROI Heads")
print("    ↓")
print("Loss calculation")

print()
print("✓ Faster R-CNN is ready for Cell 16 training")
print("=" * 70)

In [ ]:
# ============================================================
# 16. Train detector - ROBUST VERSION
# ============================================================

import os
import torch
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Verify detector exists
# ------------------------------------------------------------
if "detector" not in globals():
    raise RuntimeError(
        "Detector is not defined. "
        "Run the detector creation cell first."
    )


# ------------------------------------------------------------
# 2. Verify DataLoader exists
# ------------------------------------------------------------
if "det_train_loader" not in globals():
    raise RuntimeError(
        "det_train_loader is not defined. "
        "Run the detector dataset/DataLoader cell first."
    )


# ------------------------------------------------------------
# 3. Detector parameters
# ------------------------------------------------------------
det_params = [
    p for p in detector.parameters()
    if p.requires_grad
]

print(
    f"Trainable detector parameters: "
    f"{sum(p.numel() for p in det_params):,}"
)


# ------------------------------------------------------------
# 4. Optimizer
# ------------------------------------------------------------
det_optimizer = torch.optim.SGD(
    det_params,
    lr=DET_LR,
    momentum=0.9,
    weight_decay=1e-4,
    nesterov=True
)


# ------------------------------------------------------------
# 5. Learning-rate scheduler
# ------------------------------------------------------------
det_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    det_optimizer,
    T_max=max(int(DET_EPOCHS), 1),
    eta_min=DET_LR * 0.01
)


# ------------------------------------------------------------
# 6. AMP
# ------------------------------------------------------------
use_cuda_amp = (
    DEVICE.type == "cuda"
    and torch.cuda.is_available()
    and bool(amp_enabled)
)

if use_cuda_amp:
    det_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True
    )
else:
    det_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=False
    )

print("Device:", DEVICE)
print("AMP enabled:", use_cuda_amp)


# ------------------------------------------------------------
# 7. Checkpoint
# ------------------------------------------------------------
os.makedirs(
    CKPT_DIR,
    exist_ok=True
)

det_ckpt_path = os.path.join(
    CKPT_DIR,
    "detector_best.pt"
)


# ------------------------------------------------------------
# 8. Helper: clean Faster R-CNN targets
# ------------------------------------------------------------
def clean_detector_target(target, image_height, image_width):
    """
    Makes sure Faster R-CNN receives valid boxes and labels.

    Expected:
        boxes  -> FloatTensor[N, 4]
        labels -> Int64Tensor[N]

    Boxes are clipped to image boundaries and invalid boxes
    are removed.
    """

    boxes = target["boxes"]

    labels = target["labels"]

    # Correct dtype
    boxes = boxes.float()
    labels = labels.long()

    # --------------------------------------------------------
    # Handle completely empty target
    # --------------------------------------------------------
    if boxes.numel() == 0:
        return {
            "boxes": torch.zeros(
                (0, 4),
                dtype=torch.float32,
                device=DEVICE
            ),
            "labels": torch.zeros(
                (0,),
                dtype=torch.int64,
                device=DEVICE
            )
        }

    # Ensure shape [N, 4]
    boxes = boxes.reshape(-1, 4)

    # --------------------------------------------------------
    # Remove NaN / Inf
    # --------------------------------------------------------
    finite_mask = torch.isfinite(boxes).all(dim=1)

    boxes = boxes[finite_mask]
    labels = labels[finite_mask]

    if boxes.numel() == 0:
        return {
            "boxes": torch.zeros(
                (0, 4),
                dtype=torch.float32,
                device=DEVICE
            ),
            "labels": torch.zeros(
                (0,),
                dtype=torch.int64,
                device=DEVICE
            )
        }

    # --------------------------------------------------------
    # Clip coordinates to image boundaries
    # --------------------------------------------------------
    boxes[:, 0] = boxes[:, 0].clamp(
        0,
        max(float(image_width - 1), 0)
    )

    boxes[:, 1] = boxes[:, 1].clamp(
        0,
        max(float(image_height - 1), 0)
    )

    boxes[:, 2] = boxes[:, 2].clamp(
        0,
        float(image_width)
    )

    boxes[:, 3] = boxes[:, 3].clamp(
        0,
        float(image_height)
    )

    # --------------------------------------------------------
    # Make sure x2 > x1 and y2 > y1
    # --------------------------------------------------------
    valid_area = (
        (boxes[:, 2] > boxes[:, 0]) &
        (boxes[:, 3] > boxes[:, 1])
    )

    boxes = boxes[valid_area]
    labels = labels[valid_area]

    # --------------------------------------------------------
    # Remove invalid labels
    #
    # Faster R-CNN uses:
    #   0 = background
    #   1...N = object classes
    # --------------------------------------------------------
    valid_labels = (
        torch.isfinite(labels.float())
        & (labels >= 1)
        & (labels <= NUM_FINDINGS)
        if labels.numel() > 0
        else torch.zeros(
            (0,),
            dtype=torch.bool,
            device=labels.device
        )
    )

    boxes = boxes[valid_labels]
    labels = labels[valid_labels]

    return {
        "boxes": boxes,
        "labels": labels
    }


# ------------------------------------------------------------
# 9. Training
# ------------------------------------------------------------
best_train_loss = float("inf")

history = []


for epoch in range(int(DET_EPOCHS)):

    detector.train()

    running_loss = 0.0

    valid_batches = 0

    skipped_batches = 0

    pbar = tqdm(
        det_train_loader,
        desc=f"Detector {epoch + 1}/{DET_EPOCHS}"
    )


    for batch_idx, batch in enumerate(pbar):

        images, targets, _ = batch


        # ----------------------------------------------------
        # Move images to device
        # ----------------------------------------------------
        images = [
            img.to(
                DEVICE,
                non_blocking=True
            )
            for img in images
        ]


        # ----------------------------------------------------
        # Clean targets
        # ----------------------------------------------------
        clean_targets = []

        for img, target in zip(images, targets):

            # Image is [C,H,W]
            image_height = img.shape[-2]
            image_width = img.shape[-1]

            cleaned = clean_detector_target(
                target,
                image_height,
                image_width
            )

            clean_targets.append(cleaned)


        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------
        has_invalid = False

        for target in clean_targets:

            boxes = target["boxes"]

            if boxes.numel() > 0:

                if not torch.isfinite(boxes).all():
                    has_invalid = True
                    break

                if not (
                    boxes[:, 2] > boxes[:, 0]
                ).all():
                    has_invalid = True
                    break

                if not (
                    boxes[:, 3] > boxes[:, 1]
                ).all():
                    has_invalid = True
                    break


        if has_invalid:

            skipped_batches += 1

            pbar.set_postfix(
                status="invalid-box-skip"
            )

            continue


        # ----------------------------------------------------
        # Clear gradients BEFORE forward pass
        # ----------------------------------------------------
        det_optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Forward + loss
        # ----------------------------------------------------
        with torch.autocast(
            device_type=DEVICE.type,
            enabled=use_cuda_amp
        ):

            loss_dict = detector(
                images,
                clean_targets
            )

            loss = sum(
                value
                for value in loss_dict.values()
            )


        # ----------------------------------------------------
        # Check loss
        # ----------------------------------------------------
        if not torch.isfinite(loss):

            skipped_batches += 1

            print(
                f"\nSkipping batch {batch_idx}: "
                f"non-finite loss = {loss.detach().item()}"
            )

            continue


        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------
        if use_cuda_amp:

            det_scaler.scale(
                loss
            ).backward()

            det_scaler.unscale_(
                det_optimizer
            )

            grad_norm = torch.nn.utils.clip_grad_norm_(
                det_params,
                max_norm=5.0
            )

            det_scaler.step(
                det_optimizer
            )

            det_scaler.update()

        else:

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                det_params,
                max_norm=5.0
            )

            det_optimizer.step()


        # ----------------------------------------------------
        # Record loss
        # ----------------------------------------------------
        loss_value = float(
            loss.detach().item()
        )

        running_loss += loss_value

        valid_batches += 1


        # ----------------------------------------------------
        # Individual losses
        # ----------------------------------------------------
        loss_classifier = float(
            loss_dict["loss_classifier"]
            .detach()
            .item()
        )

        loss_box_reg = float(
            loss_dict["loss_box_reg"]
            .detach()
            .item()
        )

        loss_objectness = float(
            loss_dict["loss_objectness"]
            .detach()
            .item()
        )

        loss_rpn_box_reg = float(
            loss_dict["loss_rpn_box_reg"]
            .detach()
            .item()
        )


        # ----------------------------------------------------
        # Progress bar
        # ----------------------------------------------------
        pbar.set_postfix(
            loss=f"{loss_value:.4f}",
            cls=f"{loss_classifier:.3f}",
            box=f"{loss_box_reg:.3f}",
            obj=f"{loss_objectness:.3f}",
            rpn=f"{loss_rpn_box_reg:.3f}"
        )


    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------
    det_scheduler.step()


    # --------------------------------------------------------
    # Epoch loss
    # --------------------------------------------------------
    if valid_batches > 0:

        avg_loss = (
            running_loss /
            valid_batches
        )

    else:

        avg_loss = float("inf")


    current_lr = det_optimizer.param_groups[0]["lr"]


    print()
    print("=" * 70)

    print(
        f"Epoch {epoch + 1}/{DET_EPOCHS}"
    )

    print(
        f"Detector train loss : {avg_loss:.6f}"
    )

    print(
        f"Learning rate       : {current_lr:.8f}"
    )

    print(
        f"Valid batches       : {valid_batches}"
    )

    print(
        f"Skipped batches     : {skipped_batches}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------
    history.append({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "lr": current_lr,
        "valid_batches": valid_batches,
        "skipped_batches": skipped_batches
    })


    # --------------------------------------------------------
    # Save BEST detector
    # --------------------------------------------------------
    if (
        valid_batches > 0
        and avg_loss < best_train_loss
    ):

        best_train_loss = avg_loss

        torch.save(
            {
                "model": detector.state_dict(),

                "optimizer": (
                    det_optimizer.state_dict()
                ),

                "scheduler": (
                    det_scheduler.state_dict()
                ),

                "scaler": (
                    det_scaler.state_dict()
                    if use_cuda_amp
                    else None
                ),

                "epoch": epoch + 1,

                "best_train_loss":
                    best_train_loss,

                "img_size":
                    IMG_SIZE_DET,

                "num_classes":
                    NUM_CLASSES_DET,

                "class_names":
                    CLASS_NAMES
                    if "CLASS_NAMES" in globals()
                    else None
            },
            det_ckpt_path
        )

        print(
            f"✓ Saved BEST detector → "
            f"{det_ckpt_path}"
        )


print()
print("=" * 70)
print("Detector training finished")
print("=" * 70)

print(
    f"Best detector training loss: "
    f"{best_train_loss:.6f}"
)

print(
    f"Checkpoint: {det_ckpt_path}"
)

## 8. Optional validation mAP @ IoU 0.4

The competition metric is PASCAL VOC-style mAP at IoU > 0.4. The cell below computes a lightweight validation estimate on a configurable number of validation images.

It is intentionally optional because detector inference can be slow. The ground-truth boxes are already in the canonical 1024×1024 coordinate system, and predictions are converted back to that same coordinate system before IoU is calculated.


In [ ]:
# ============================================================
# 17. Lightweight mAP@0.4 evaluation
# ============================================================
def compute_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter = (
        max(0.0, x2 - x1) *
        max(0.0, y2 - y1)
    )

    area_a = (
        max(0.0, box_a[2] - box_a[0]) *
        max(0.0, box_a[3] - box_a[1])
    )

    area_b = (
        max(0.0, box_b[2] - box_b[0]) *
        max(0.0, box_b[3] - box_b[1])
    )

    union = area_a + area_b - inter

    return inter / max(union, 1e-9)


def voc_ap(rec, prec):
    mrec = np.concatenate(
        ([0.0], rec, [1.0])
    )

    mpre = np.concatenate(
        ([0.0], prec, [0.0])
    )

    for i in range(
        len(mpre) - 1,
        0,
        -1
    ):
        mpre[i - 1] = max(
            mpre[i - 1],
            mpre[i]
        )

    idx = np.where(
        mrec[1:] != mrec[:-1]
    )[0]

    return np.sum(
        (
            mrec[idx + 1] -
            mrec[idx]
        ) *
        mpre[idx + 1]
    )


@torch.no_grad()
def evaluate_map_04(
    model,
    val_df,
    max_images=300,
    score_threshold=0.05
):
    model.eval()

    eval_ids = (
        val_df["image_id"]
        .drop_duplicates()
        .tolist()
    )

    if max_images is not None:
        eval_ids = eval_ids[
            :max_images
        ]

    gt = {}

    subset = val_df[
        val_df["image_id"].isin(
            eval_ids
        )
    ]

    for image_id, g in subset.groupby(
        "image_id"
    ):
        entries = {}

        for cls, cg in g[
            g["class_id"] != NO_FINDING_ID
        ].groupby("class_id"):
            entries[int(cls)] = (
                cg[
                    [
                        "x_min",
                        "y_min",
                        "x_max",
                        "y_max"
                    ]
                ]
                .values
                .astype(float)
                .tolist()
            )

        gt[image_id] = entries

    detections = {
        c: []
        for c in range(NUM_FINDINGS)
    }

    n_gt = {
        c: 0
        for c in range(NUM_FINDINGS)
    }

    for image_id in tqdm(
        eval_ids,
        desc="mAP validation"
    ):
        img = load_image_any(
            find_image_path(
                TRAIN_IMG_DIR,
                image_id
            )
        )

        canvas, scale, left, top = (
            letterbox_resize(
                img,
                IMG_SIZE_DET
            )
        )

        x = to_tensor_raw(canvas).to(
            DEVICE
        )

        pred = model([x])[0]

        boxes = (
            pred["boxes"]
            .detach()
            .cpu()
            .numpy()
        )

        scores = (
            pred["scores"]
            .detach()
            .cpu()
            .numpy()
        )

        labels = (
            pred["labels"]
            .detach()
            .cpu()
            .numpy()
            - 1
        )

        for box, score, label in zip(
            boxes,
            scores,
            labels
        ):
            if score < score_threshold:
                continue

            if not (
                0 <= label <
                NUM_FINDINGS
            ):
                continue

            b = box.copy()

            b[[0, 2]] = (
                b[[0, 2]] - left
            ) / scale

            b[[1, 3]] = (
                b[[1, 3]] - top
            ) / scale

            # Convert actual loaded-image coordinates back to the
            # canonical 1024×1024 coordinate system used by GT.
            h0, w0 = img.shape[:2]
            b[[0, 2]] *= float(ANNOTATION_IMAGE_SIZE) / max(w0, 1)
            b[[1, 3]] *= float(ANNOTATION_IMAGE_SIZE) / max(h0, 1)

            detections[
                int(label)
            ].append(
                (
                    image_id,
                    float(score),
                    b.tolist()
                )
            )

    for image_id, classes in gt.items():
        for cls, boxes in classes.items():
            n_gt[cls] += len(boxes)

    aps = {}

    for cls in range(NUM_FINDINGS):
        dets = sorted(
            detections[cls],
            key=lambda z: z[1],
            reverse=True
        )

        matched = {
            image_id:
            np.zeros(
                len(
                    gt.get(
                        image_id,
                        {}
                    ).get(cls, [])
                ),
                dtype=bool
            )
            for image_id in eval_ids
        }

        tp = np.zeros(
            len(dets)
        )

        fp = np.zeros(
            len(dets)
        )

        for i, (
            image_id,
            score,
            box
        ) in enumerate(dets):
            gt_boxes = (
                gt.get(
                    image_id,
                    {}
                ).get(cls, [])
            )

            if len(gt_boxes) == 0:
                fp[i] = 1
                continue

            ious = np.array([
                compute_iou(
                    box,
                    gbox
                )
                for gbox in gt_boxes
            ])

            best_idx = int(
                np.argmax(ious)
            )

            best_iou = ious[
                best_idx
            ]

            if (
                best_iou >= 0.4 and
                not matched[
                    image_id
                ][best_idx]
            ):
                tp[i] = 1
                matched[
                    image_id
                ][best_idx] = True
            else:
                fp[i] = 1

        if n_gt[cls] == 0:
            continue

        cum_tp = np.cumsum(tp)
        cum_fp = np.cumsum(fp)

        rec = (
            cum_tp /
            max(n_gt[cls], 1)
        )

        prec = (
            cum_tp /
            np.maximum(
                cum_tp + cum_fp,
                1e-9
            )
        )

        aps[cls] = voc_ap(
            rec,
            prec
        )

    map_value = (
        float(
            np.mean(
                list(aps.values())
            )
        )
        if aps
        else 0.0
    )

    return map_value, aps


# Uncomment after training:
# map04, aps04 = evaluate_map_04(
#     detector,
#     val_df_split,
#     max_images=300
# )
# print("Validation mAP@0.4:", map04)
# print("Per-class AP:", aps04)


In [ ]:
# ============================================================
# 18. Co-occurrence rescoring
# ============================================================
def build_cooccurrence_matrix(df):
    mat = np.zeros(
        (NUM_FINDINGS, NUM_FINDINGS),
        dtype=np.float64
    )

    counts = np.zeros(
        NUM_FINDINGS,
        dtype=np.float64
    )

    df = df[
        df["class_id"] != NO_FINDING_ID
    ]

    for _, g in df.groupby(
        "image_id"
    ):
        classes = sorted(
            set(
                g["class_id"]
                .astype(int)
            )
        )

        for c in classes:
            if 0 <= c < NUM_FINDINGS:
                counts[c] += 1

        for i in classes:
            for j in classes:
                if (
                    i != j and
                    0 <= i < NUM_FINDINGS and
                    0 <= j < NUM_FINDINGS
                ):
                    mat[i, j] += 1

    return np.divide(
        mat,
        counts[:, None],
        out=np.zeros_like(mat),
        where=counts[:, None] > 0
    )


COOC = build_cooccurrence_matrix(
    train_df_split
)


def rescore_with_cooccurrence(
    scores,
    labels,
    boost=0.05,
    trigger_thr=0.50
):
    if len(scores) == 0:
        return scores

    scores = np.asarray(
        scores,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=int
    )

    present = np.zeros(
        NUM_FINDINGS
    )

    for label, score in zip(
        labels,
        scores
    ):
        if (
            0 <= label <
            NUM_FINDINGS
        ):
            present[label] = max(
                present[label],
                score
            )

    new_scores = scores.copy()

    for i, label in enumerate(
        labels
    ):
        if not (
            0 <= label <
            NUM_FINDINGS
        ):
            continue

        supports = [
            COOC[j, label]
            for j in range(NUM_FINDINGS)
            if (
                j != label and
                present[j] >= trigger_thr
            )
        ]

        partner_support = max(
            supports,
            default=0.0
        )

        new_scores[i] = min(
            1.0,
            scores[i] +
            boost * partner_support
        )

    return new_scores.tolist()


print(
    "Co-occurrence matrix shape:",
    COOC.shape
)


## 9. Inference and submission

The detector predictions are converted from the 512px letterboxed coordinate system back to the canonical 1024×1024 competition-image coordinates.

For an image with no accepted detection, the competition format is:

`14 1.0 0 0 1 1`


In [ ]:
# ============================================================
# 19. Load best checkpoints
# ============================================================
if not os.path.exists(clf_ckpt_path):
    raise FileNotFoundError(
        f"Classifier checkpoint not found: {clf_ckpt_path}. "
        "Run classifier training first."
    )

if not os.path.exists(det_ckpt_path):
    raise FileNotFoundError(
        f"Detector checkpoint not found: {det_ckpt_path}. "
        "Run detector training first."
    )

clf_checkpoint = torch.load(
    clf_ckpt_path,
    map_location=DEVICE
)

det_checkpoint = torch.load(
    det_ckpt_path,
    map_location=DEVICE
)

classifier.load_state_dict(
    clf_checkpoint["model"]
)

detector.load_state_dict(
    det_checkpoint["model"]
)

classifier.eval()
detector.eval()

print("Loaded classifier:", clf_ckpt_path)
print("Loaded detector:", det_ckpt_path)
print("Classifier best validation loss:", clf_checkpoint.get("best_val_loss", "not stored"))
print("Detector best training loss:", det_checkpoint.get("best_train_loss", "not stored"))


In [ ]:
# ============================================================
# 20. Detection inference helper
# ============================================================
NO_FINDING_LINE = (
    "14 1.0 0 0 1 1"
)


def nms_numpy(
    boxes,
    scores,
    iou_thr=0.4
):
    if len(boxes) == 0:
        return []

    boxes = np.asarray(
        boxes,
        dtype=float
    )

    scores = np.asarray(
        scores,
        dtype=float
    )

    order = scores.argsort()[::-1]

    keep = []

    while len(order):
        i = order[0]

        keep.append(i)

        if len(order) == 1:
            break

        rest = order[1:]

        xx1 = np.maximum(
            boxes[i, 0],
            boxes[rest, 0]
        )

        yy1 = np.maximum(
            boxes[i, 1],
            boxes[rest, 1]
        )

        xx2 = np.minimum(
            boxes[i, 2],
            boxes[rest, 2]
        )

        yy2 = np.minimum(
            boxes[i, 3],
            boxes[rest, 3]
        )

        inter = (
            np.maximum(
                0,
                xx2 - xx1
            )
            *
            np.maximum(
                0,
                yy2 - yy1
            )
        )

        area_i = (
            max(
                0,
                boxes[i, 2] -
                boxes[i, 0]
            )
            *
            max(
                0,
                boxes[i, 3] -
                boxes[i, 1]
            )
        )

        area_rest = (
            np.maximum(
                0,
                boxes[rest, 2] -
                boxes[rest, 0]
            )
            *
            np.maximum(
                0,
                boxes[rest, 3] -
                boxes[rest, 1]
            )
        )

        iou = inter / np.maximum(
            area_i +
            area_rest -
            inter,
            1e-9
        )

        order = rest[
            iou < iou_thr
        ]

    return keep


@torch.no_grad()
def detect_one_image(
    img,
    score_thresh=DET_SCORE_THRESHOLD
):
    h0, w0 = img.shape[:2]

    canvas, scale, left, top = (
        letterbox_resize(
            img,
            IMG_SIZE_DET
        )
    )

    x = to_tensor_raw(canvas).to(
        DEVICE
    )

    pred = detector([x])[0]

    boxes = (
        pred["boxes"]
        .detach()
        .cpu()
        .numpy()
    )

    scores = (
        pred["scores"]
        .detach()
        .cpu()
        .numpy()
    )

    labels = (
        pred["labels"]
        .detach()
        .cpu()
        .numpy()
        - 1
    )

    keep = (
        scores >=
        score_thresh
    )

    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    if len(boxes) == 0:
        return [], [], []

    # Undo letterbox transformation.
    boxes[:, [0, 2]] = (
        boxes[:, [0, 2]] -
        left
    ) / scale

    boxes[:, [1, 3]] = (
        boxes[:, [1, 3]] -
        top
    ) / scale

    boxes[:, [0, 2]] = np.clip(
        boxes[:, [0, 2]],
        0,
        w0
    )

    boxes[:, [1, 3]] = np.clip(
        boxes[:, [1, 3]],
        0,
        h0
    )

    # Competition PNGs are 1024×1024. Convert from the actual loaded
    # image coordinates to the canonical submission coordinate system.
    boxes[:, [0, 2]] *= float(ANNOTATION_IMAGE_SIZE) / max(w0, 1)
    boxes[:, [1, 3]] *= float(ANNOTATION_IMAGE_SIZE) / max(h0, 1)
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, ANNOTATION_IMAGE_SIZE - 1)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, ANNOTATION_IMAGE_SIZE - 1)

    final_boxes = []
    final_scores = []
    final_labels = []

    # Class-wise NMS.
    for cls in sorted(
        set(labels.tolist())
    ):
        idx = np.where(
            labels == cls
        )[0]

        keep_idx = nms_numpy(
            boxes[idx],
            scores[idx],
            iou_thr=DET_NMS_THRESHOLD
        )

        for k in keep_idx:
            original_idx = idx[k]

            final_boxes.append(
                boxes[original_idx]
            )

            final_scores.append(
                float(
                    scores[
                        original_idx
                    ]
                )
            )

            final_labels.append(
                int(
                    labels[
                        original_idx
                    ]
                )
            )

    order = np.argsort(
        final_scores
    )[::-1][:MAX_DETECTIONS]

    return (
        [
            final_boxes[i].tolist()
            for i in order
        ],
        [
            final_scores[i]
            for i in order
        ],
        [
            final_labels[i]
            for i in order
        ]
    )


In [ ]:
# ============================================================
# 21. Generate submission.csv
# ============================================================
# Use sample_submission.csv as the authoritative submission order.
sample_df = pd.read_csv(SAMPLE_SUBMISSION)

assert "image_id" in sample_df.columns, (
    "sample_submission.csv must contain image_id."
)

test_image_ids = (
    sample_df["image_id"]
    .astype(str)
    .tolist()
)

assert len(test_image_ids) == sample_df["image_id"].nunique(), (
    "sample_submission.csv contains duplicate image_id values."
)

print("Test images required by sample submission:", len(test_image_ids))

# The detector can already return zero detections for normal images.
# Keep the triage gate OFF by default because a weak/short classifier
# could otherwise suppress true abnormalities before the detector sees them.
USE_TRIAGE_GATE = False
TRIAGE_THRESHOLD = 0.50

rows = []

for image_id in tqdm(
    test_image_ids,
    desc="Generating submission"
):
    img = load_image_any(
        find_image_path(
            TEST_IMG_DIR,
            image_id
        )
    )

    run_detector = True

    if USE_TRIAGE_GATE:
        canvas, _, _, _ = (
            letterbox_resize(
                img,
                IMG_SIZE_CLF
            )
        )

        x = to_tensor(
            canvas
        ).unsqueeze(0).to(
            DEVICE
        )

        with torch.no_grad():
            out = classifier(x)

            p_abnormal = torch.sigmoid(
                out["logit"]
            ).item()

        run_detector = (
            p_abnormal >= TRIAGE_THRESHOLD
        )

    if not run_detector:
        rows.append({
            "image_id": image_id,
            "PredictionString": NO_FINDING_LINE
        })
        continue

    boxes, scores, labels = detect_one_image(img)

    if len(boxes) == 0:
        rows.append({
            "image_id": image_id,
            "PredictionString": NO_FINDING_LINE
        })
        continue

    scores = rescore_with_cooccurrence(
        scores,
        labels,
        boost=0.05
    )

    parts = []

    for box, score, label in zip(
        boxes,
        scores,
        labels
    ):
        x1, y1, x2, y2 = [
            int(round(v))
            for v in box
        ]

        x1 = int(np.clip(x1, 0, ANNOTATION_IMAGE_SIZE - 1))
        y1 = int(np.clip(y1, 0, ANNOTATION_IMAGE_SIZE - 1))
        x2 = int(np.clip(x2, x1 + 1, ANNOTATION_IMAGE_SIZE))
        y2 = int(np.clip(y2, y1 + 1, ANNOTATION_IMAGE_SIZE))

        # Challenge class IDs are 0-13 for findings.
        if not (0 <= int(label) < NUM_FINDINGS):
            continue

        parts.append(
            f"{int(label)} "
            f"{float(np.clip(score, 0.0, 1.0)):.4f} "
            f"{x1} {y1} "
            f"{x2} {y2}"
        )

    prediction = (
        " ".join(parts)
        if parts
        else NO_FINDING_LINE
    )

    rows.append({
        "image_id": image_id,
        "PredictionString": prediction
    })

submission = pd.DataFrame(
    rows,
    columns=[
        "image_id",
        "PredictionString"
    ]
)

submission_path = os.path.join(
    WORK_DIR,
    "submission.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print("Saved submission:", submission_path)
print("Rows:", len(submission))
display(submission.head(10))


In [ ]:
# ============================================================
# 22. Final submission checks
# ============================================================
assert list(submission.columns) == [
    "image_id",
    "PredictionString"
]

assert len(submission) == len(test_image_ids)

assert submission["image_id"].astype(str).tolist() == [
    str(x) for x in test_image_ids
]

assert submission["image_id"].is_unique
assert submission["PredictionString"].notna().all()

# Validate each prediction string.
valid_class_ids = set(range(NUM_FINDINGS)) | {NO_FINDING_ID}

def validate_prediction_string(s):
    if not isinstance(s, str) or not s.strip():
        return False

    tokens = s.split()

    # No-finding prediction has exactly 6 tokens.
    if s.strip() == NO_FINDING_LINE:
        return True

    if len(tokens) % 6 != 0:
        return False

    for i in range(0, len(tokens), 6):
        cls = int(tokens[i])
        score = float(tokens[i + 1])
        x1, y1, x2, y2 = map(int, tokens[i + 2:i + 6])

        if cls not in valid_class_ids or cls == NO_FINDING_ID:
            return False
        if not (0.0 <= score <= 1.0):
            return False
        if not (0 <= x1 < x2 <= ANNOTATION_IMAGE_SIZE):
            return False
        if not (0 <= y1 < y2 <= ANNOTATION_IMAGE_SIZE):
            return False

    return True

assert submission["PredictionString"].map(
    validate_prediction_string
).all()

assert os.path.exists(submission_path)

print("✓ Submission columns are correct")
print("✓ Image IDs and order match sample_submission.csv")
print("✓ PredictionString format is valid")
print("✓ All boxes are within 1024×1024 coordinates")
print("File:", submission_path)
print(
    "Size (MB):",
    round(
        os.path.getsize(submission_path) / 1024**2,
        2
    )
)


## 11. Final tuning notes

### First smoke-test run
Use:
```python
MAX_TRAIN_IMAGES = 500
CLF_EPOCHS = 1
DET_EPOCHS = 1
```

This is only to verify that the complete pipeline works.

### Full training run
After the smoke test succeeds, change:
```python
MAX_TRAIN_IMAGES = None
CLF_EPOCHS = 3
DET_EPOCHS = 5
```

For a stronger competition run, increase the epochs if GPU/time permits.

### CUDA out-of-memory
Try:
```python
DET_BATCH = 1
IMG_SIZE_DET = 448
NUM_WORKERS = 1
```

### Pretrained weights
Keep:
```python
USE_PRETRAINED = True
```
If the Kaggle environment cannot obtain ImageNet weights, the notebook automatically falls back to random initialization.

### Important
Do **not** enable the triage gate for the first final submission:
```python
USE_TRIAGE_GATE = False
```
The detector itself handles no-detection images. The classifier is retained as a triage model, but a weak classifier should not suppress detector predictions.

### Final file
```text
/kaggle/working/submission.csv
```

Submission format:
```text
image_id,PredictionString
```

Class IDs:
- `0–13`: 14 abnormalities
- `14`: `No finding`

Coordinates are emitted in the challenge's canonical 1024×1024 image coordinate system.


## References

- AMIA Public Challenge 2026 — Kaggle competition description.
- VinBigData Chest X-ray Abnormalities Detection — task/dataset description.
- Huang et al., Densely Connected Convolutional Networks.
- Ren et al., Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks.
